# Research: Stock Recommendations & Data Mining (S&P 1500+ Super-Charged)

This local notebook replicates the **entire** data mining, technical analysis, and vectorization logic from the backend. 
- It actively scrapes Wikipedia for the S&P 500, S&P 400 (Mid-Cap), S&P 600 (Small-Cap), and Nasdaq 100 to yield a perfect **1,500+ US Equity Universe**.
- Uses Vectorized Pandas Operations to instantly generate massive historical sliding windows (MACD, RSI, Volatility).
- Connects to Yahoo Finance APIs to calculate live Options Greeks (Black-Scholes) and Fundamental Company metrics.

In [ ]:
!pip install yfinance matplotlib seaborn scipy lxml html5lib requests tqdm

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import math
import time
import os
import requests
from scipy.stats import norm
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib style for financial charts
plt.style.use('dark_background')
%matplotlib inline

# Connect to the local database for Questionnaire access
DB_PATH = "backend/stock_recommender.db"
conn = sqlite3.connect(DB_PATH)

## 1. Massive Data Extraction (S&P 1500 Universe)
We scrape the massive universe of 1500+ stocks safely, then pull their max historical daily closes. (Includes CSV Caching to save time on reruns!)

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) stock-research-toolkit/1.0'}

def scrape_wiki_tickers(url, table_idx, ticker_col):
    try:
        res = requests.get(url, headers=headers)
        table = pd.read_html(res.text)[table_idx]
        tickers = table[ticker_col].str.replace('.', '-', regex=False).tolist()
        time.sleep(1)
        return tickers
    except:
        print(f"Failed to scrape {url}")
        return []

print("Scraping S&P 500 (Large-Cap)...")
sp500 = scrape_wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', 0, 'Symbol')

print("Scraping S&P 400 (Mid-Cap)...")
sp400 = scrape_wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_400_companies', 0, 'Symbol')

print("Scraping S&P 600 (Small-Cap)...")
sp600 = scrape_wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_600_companies', 0, 'Symbol')

print("Scraping Nasdaq 100...")
ndx = scrape_wiki_tickers('https://en.wikipedia.org/wiki/Nasdaq-100', 4, 'Ticker')

EQUITY_ETFS = ["VOO", "QQQ", "VTI", "VXUS", "VGT", "ARKK", "VNQ", "VWO"]
BOND_ETFS = ["BND", "SGOV", "TLT", "TIPS"]

# Combine all identified tickers into a 1500+ Mega-list
ALL_TICKERS = list(set(sp500 + sp400 + sp600 + ndx + EQUITY_ETFS + BOND_ETFS))
print(f"\nTotal Unique S&P 1500+ Universe Assembled: {len(ALL_TICKERS)}")

# Intelligent Caching System
PRICE_FILE = "sp1500_price_matrix.csv"
VOLUME_FILE = "sp1500_volume_matrix.csv"

if os.path.exists(PRICE_FILE) and os.path.exists(VOLUME_FILE):
    print("\n=> Found cached historical data! Loading directly from CSVs to save time...")
    price_matrix = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
    volume_matrix = pd.read_csv(VOLUME_FILE, index_col=0, parse_dates=True)
else:
    print("\n=> Fetching 'max' history from Yahoo Finance (Warning: Downloading millions of data points will take a few minutes)...")
    raw_data = yf.download(ALL_TICKERS, period="max", group_by="ticker", auto_adjust=True, progress=True)

    # Build matrices natively using cross-section to avoid DataFrame fragmentation
    if isinstance(raw_data.columns, pd.MultiIndex):
        price_matrix = raw_data.xs('Close', axis=1, level=1).copy()
        volume_matrix = raw_data.xs('Volume', axis=1, level=1).copy()
    else:
        price_matrix = pd.DataFrame({ALL_TICKERS[0]: raw_data['Close']})
        volume_matrix = pd.DataFrame({ALL_TICKERS[0]: raw_data['Volume']})
        
    # Forward fill weekends/holidays up to today
    price_matrix = price_matrix.resample('D').ffill()
    volume_matrix = volume_matrix.resample('D').ffill()
    
    # Save for future runs
    price_matrix.to_csv(PRICE_FILE)
    volume_matrix.to_csv(VOLUME_FILE)

daily_returns = price_matrix.pct_change().dropna(how='all')
correlation_matrix = daily_returns.corr()

print(f"\nSuccessfully loaded historical price matrix: {price_matrix.shape[0]} Days x {price_matrix.shape[1]} Tickers")


## 2. Vectorized Historical Technical Indicators
Compute sliding-window indicators (MACD, RSI, Bollinger Bands, Volatility) for the entire dataset matrix instantly using pure Pandas vectorization.

In [ ]:
print("Calculating Historical Technical Indicators (MACD, RSI, BB, Volatility) for 1500+ Stocks...")

# EMA and SMA for MACD logic
ema_12 = price_matrix.ewm(span=12, adjust=False).mean()
ema_26 = price_matrix.ewm(span=26, adjust=False).mean()
macd_line = ema_12 - ema_26
macd_signal = macd_line.ewm(span=9, adjust=False).mean()

# RSI (14-day) Sliding window across entire matrix natively
delta = price_matrix.diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
rsi_14 = 100 - (100 / (1 + rs))

# Bollinger Bands (20-day, 2 std)
bb_std = price_matrix.rolling(window=20).std()
bb_upper = price_matrix.rolling(window=20).mean() + (bb_std * 2)
bb_lower = price_matrix.rolling(window=20).mean() - (bb_std * 2)

# Volatilities (Annualised) moving windows
vol_30d = daily_returns.rolling(window=30).std() * np.sqrt(252)
vol_90d = daily_returns.rolling(window=90).std() * np.sqrt(252)
vol_1yr = daily_returns.rolling(window=252).std() * np.sqrt(252)

# Rolling Returns
return_1yr = price_matrix.pct_change(periods=252)

print("Historical feature matrices generated instantaneously!")

## 3. Build Master Research DataFrame (Heavy API Loop)
We will now loop through all 1500+ tickers to query fundamental info & Options Greeks. 
**NOTE:** This automatically saves a giant CSV caching the results, so you never have to waste an hour running this twice!

In [ ]:
MASTER_FILE = "sp1500_master_research_dataset.csv"

if os.path.exists(MASTER_FILE):
    print("=> Found cached Master Research Dataset! Loading directly from CSV...")
    master_research_df = pd.read_csv(MASTER_FILE, index_col=0)
    
else:
    print("=> No cached master dataset found. Beginning the heavy 1-hour API extraction loop...")
    RISK_FREE_RATE = 0.045
    company_data = []

    # If you don't have an hour to wait natively, change ALL_TICKERS to ALL_TICKERS[:50] to test a smaller subset!
    for ticker in tqdm(ALL_TICKERS, desc="Fetching Fundamental & Options Data"):
        try:
            t_obj = yf.Ticker(ticker)
            info = t_obj.info
            
            # We only process valid stocks that return info
            if 'symbol' not in info and 'shortName' not in info:
                continue
                
            company_record = {
                'ticker': ticker,
                'sector': info.get('sector', 'Unknown'),
                'industry': info.get('industry', 'Unknown'),
                'market_cap': info.get('marketCap', 0),
                'pe_ratio': info.get('trailingPE', np.nan),
                'beta': info.get('beta', np.nan),
                'dividend_yield': info.get('dividendYield', 0),
                
                # Detailed Company Information
                'long_description': info.get('longBusinessSummary', ''),
                'website': info.get('website', ''),
                'city': info.get('city', ''),
                'state': info.get('state', ''),
                'country': info.get('country', ''),
                'full_time_employees': info.get('fullTimeEmployees', np.nan),
                
                # Financial / Quarterly Reports Context (TTM Metrics)
                'total_revenue': info.get('totalRevenue', np.nan),
                'revenue_growth': info.get('revenueGrowth', np.nan),
                'gross_margins': info.get('grossMargins', np.nan),
                'operating_margins': info.get('operatingMargins', np.nan),
                'ebitda': info.get('ebitda', np.nan),
                'free_cashflow': info.get('freeCashflow', np.nan),
                'total_cash': info.get('totalCash', np.nan),
                'total_debt': info.get('totalDebt', np.nan)
            }
            
            # Options Greeks (Black-Scholes Approximation)
            try:
                exps = t_obj.options
                if exps:
                    nearest_exp = exps[0]
                    opt_chain = t_obj.option_chain(nearest_exp)
                    calls = opt_chain.calls
                    
                    if not calls.empty and 'currentPrice' in info:
                        current_price = info['currentPrice']
                        atm_call = calls.iloc[(calls['strike'] - current_price).abs().argsort()[:1]].iloc[0]
                        
                        S = current_price
                        K = atm_call['strike']
                        T = 30 / 365.0  # Approx 1 month
                        r = RISK_FREE_RATE
                        sigma = atm_call.get('impliedVolatility', 0.2)
                        
                        if sigma > 0:
                            d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
                            delta = norm.cdf(d1)
                            gamma = norm.pdf(d1) / (S * sigma * math.sqrt(T))
                            vega = S * norm.pdf(d1) * math.sqrt(T) / 100
                            
                            company_record.update({
                                'atm_strike': K,
                                'implied_vol': sigma,
                                'delta': delta,
                                'gamma': gamma,
                                'vega': vega
                            })
            except:
                pass # Options chain failed or not available
            
            company_data.append(company_record)
            time.sleep(0.3)  # Anti-Rate Limit Delay (crucial for 1500+ loops)
        except Exception as e:
            pass # Ignore individual failed ticker fetches
    
    company_df = pd.DataFrame(company_data).set_index('ticker')
    
    current_metrics = pd.DataFrame({
        'latest_price': price_matrix.iloc[-1],
        'avg_30d_volume': volume_matrix.rolling(30).mean().iloc[-1],
        'rsi_14': rsi_14.iloc[-1],
        'macd': macd_line.iloc[-1],
        'macd_signal': macd_signal.iloc[-1],
        'volatility_30d': vol_30d.iloc[-1],
        'volatility_1yr': vol_1yr.iloc[-1],
        'return_1yr': return_1yr.iloc[-1]
    })
    
    master_research_df = current_metrics.join(company_df)
    
    # Save the fully joined dataframe securely to a CSV to avoid ever having to run this loop again tomorrow
    master_research_df.to_csv(MASTER_FILE)

print(f"\nMaster Research Dataset Successfully Loaded: {master_research_df.shape[0]} Stocks x {master_research_df.shape[1]} Features")
display(master_research_df.head())


## 4. Giant Universe Recommender Engine Research
We can now find alternative stocks accurately utilizing 1500+ equities and precise metrics (RSI, Options Implied Vol, etc).

In [ ]:
def find_similar_stocks(target_ticker, top_n=5, method="correlation"):
    if target_ticker not in correlation_matrix.columns:
        return f"Ticker {target_ticker} not in database."
        
    if method == "correlation":
        corrs = correlation_matrix[target_ticker].sort_values(ascending=False)
        return pd.DataFrame({'Correlation': corrs[corrs.index != target_ticker].head(top_n)})
        
    elif method == "features":
        # Use the mega-dataset features here!
        feats = master_research_df[['rsi_14', 'macd', 'volatility_30d', 'return_1yr']].dropna()
        if target_ticker not in feats.index:
            return f"Missing features for {target_ticker}"
        
        normalized = (feats - feats.mean()) / feats.std()
        target_vec = normalized.loc[target_ticker]
        
        distances = np.sqrt(((normalized - target_vec)**2).sum(axis=1))
        distances = distances.sort_values()
        similar = distances[distances.index != target_ticker].head(top_n)
        
        return master_research_df.loc[similar.index, ['sector', 'latest_price', 'rsi_14', 'implied_vol']]

print("Top 5 correlated S&P 1500+ stocks to AAPL:")
display(find_similar_stocks("AAPL", method="correlation"))

print("\nTop 5 stocks with identical MACD/RSI/Volatility technical profiles to MSFT:")
display(find_similar_stocks("MSFT", method="features"))